# VisuoGrasp V3.3 — demo video recording add-on

Append the two code cells below to your already initialized V3.3 Jupyter Notebook. Do not Run All.

In [ ]:
# VisuoGrasp V3.3 — Cell 1: video demo utilities (no training)
# Add AFTER the existing V3.3 cells. Run only these demo cells, NOT Run All.
from pathlib import Path
import gc, json
import numpy as np
import pandas as pd
import cv2
import imageio.v2 as imageio

DEMO_DIR = ROOT / 'videos' / 'v33_final_demos'
DEMO_DIR.mkdir(parents=True, exist_ok=True)
DEMO_FPS = 10                 # 1 video frame for 2 simulator steps; real-time playback
DEMO_FRAME_EVERY = 2
DEMO_GIFS = True              # GIF only for Level A/Level B nominal successes and A nominal failure

# Six pairs: successes and failures for Train (nominal), Test-1, Test-2,
# plus two Level B examples. Each replay uses the same seed/shape/episode as final_eval.
DEMO_CASES = [
    ('A_train_success',  'V3_A_SAC_M2', 'nominal', True,  ['box','sphere','cylinder','ellipsoid','capsule','ring']),
    ('A_train_failure',  'V3_A_SAC_M2', 'nominal', False, ['ring','capsule','ellipsoid','sphere','box','cylinder']),
    ('A_test1_success',  'V3_A_SAC_M2', 'test1',   True,  ['box','cylinder','sphere','ellipsoid','capsule','ring']),
    ('A_test1_failure',  'V3_A_SAC_M2', 'test1',   False, ['ring','capsule','ellipsoid','sphere','box','cylinder']),
    ('A_test2_success',  'V3_A_SAC_M2', 'test2',   True,  ['mug','hex_prism','l_shape','cone','bottle']),
    ('A_test2_failure',  'V3_A_SAC_M2', 'test2',   False, ['bottle','cone','mug','l_shape','hex_prism']),
    ('B_train_success',  'V3_B_SAC_M2', 'nominal', True,  ['box','cylinder','sphere','ellipsoid','capsule','ring']),
    ('B_train_failure',  'V3_B_SAC_M2', 'nominal', False, ['ring','capsule','sphere','ellipsoid','box','cylinder']),
]


def choose_demo(tag, condition, want_success, shape_order):
    """Pick a real recorded final-eval episode, not a fabricated successful demonstration."""
    candidates = []
    for seed in (33, 22, 11):
        path = EVAL_ROOT / f'{tag}_seed{seed}_best_episodes.csv'
        if not path.is_file():
            continue
        df = pd.read_csv(path)
        df = df.loc[(df['condition'] == condition) &
                    (df['gt_success'].astype(int) == int(want_success))].copy()
        if df.empty:
            continue
        df['train_seed_demo'] = seed
        candidates.append(df)
    if not candidates:
        return None
    data = pd.concat(candidates, ignore_index=True)
    rank = {name: i for i, name in enumerate(shape_order)}
    data['shape_rank'] = data['shape'].map(rank).fillna(len(rank))
    # For failure demos prefer a real drop, then a lift without correct pad contacts.
    if not want_success and 'outcome' in data.columns:
        outcomes = {'lifted_then_dropped': 0, 'held_without_pad_pair': 1,
                    'partial_lift': 2, 'never_lifted': 3}
        data['failure_rank'] = data['outcome'].map(outcomes).fillna(4)
        data = data.sort_values(['failure_rank', 'shape_rank', 'episode'])
    else:
        data = data.sort_values(['shape_rank', 'episode'])
    return data.iloc[0].to_dict()


def _camera_bgr(e, name, width=320):
    """RGB-only render: avoids extra depth-noise draws and extra detector updates."""
    e.cv.renderer.update_scene(e.data, camera=name)
    rgb = e.cv.renderer.render().copy()
    bgr = cv2.cvtColor(np.asarray(rgb, dtype=np.uint8), cv2.COLOR_RGB2BGR)
    return cv2.resize(bgr, (width, width), interpolation=cv2.INTER_LINEAR)


def _demo_frame(e, info, step, title):
    left = _camera_bgr(e, 'side')
    right = _camera_bgr(e, 'wrist')
    w = 640
    out = np.full((480, w, 3), (29, 30, 35), dtype=np.uint8)
    out[:320, :320] = left
    out[:320, 320:640] = right
    cv2.rectangle(out, (0, 0), (640, 26), (20, 20, 20), -1)
    cv2.putText(out, 'SIDE CAMERA', (12, 18), cv2.FONT_HERSHEY_SIMPLEX, .52, (250, 250, 250), 1, cv2.LINE_AA)
    cv2.putText(out, 'WRIST CAMERA', (335, 18), cv2.FONT_HERSHEY_SIMPLEX, .52, (250, 250, 250), 1, cv2.LINE_AA)
    gt = np.asarray(info.get('gt_contacts_EVAL_ONLY', np.zeros(4)), dtype=int)
    cv = np.asarray(info['cv_contacts'], dtype=int)
    pp = np.asarray(info['cv_prob'], dtype=float)
    height = float(info['height_gain_m']) * 100
    gt_ok = int(info.get('gt_success_EVAL_ONLY', False))
    cv_ok = int(info['cv_success'])
    lines = [
        title[:67],
        f't={step / CFG.policy_hz:4.1f}s   lift={height:5.1f}cm   GT success={gt_ok}   CV success={cv_ok}',
        'GT pad contact (thumb,f1,f2,f3): ' + ''.join(map(str, gt)),
        'D2 confirmed contact:              ' + ''.join(map(str, cv)),
        'D2 probabilities:  ' + '  '.join(f'{x:.2f}' for x in pp),
    ]
    for j, text in enumerate(lines):
        color = (235, 235, 235) if j < 2 else (170, 225, 230)
        cv2.putText(out, text, (12, 340 + 27*j), cv2.FONT_HERSHEY_SIMPLEX,
                    .53 if j < 2 else .49, color, 1, cv2.LINE_AA)
    return out


def record_v33_demo(case, overwrite=False):
    label, tag, condition, want_success, shape_order = case
    target = DEMO_DIR / f'{label}.mp4'
    metadata_path = DEMO_DIR / f'{label}.json'
    if target.is_file() and target.stat().st_size > 0 and metadata_path.is_file() and not overwrite:
        print('[SKIP]', target.name, 'already exists')
        return json.loads(metadata_path.read_text(encoding='utf-8')) if metadata_path.exists() else None

    selected = choose_demo(tag, condition, want_success, shape_order)
    if selected is None:
        print('[NO MATCH]', label, '- no matching episode in final_eval')
        return None
    train_seed = int(selected['train_seed_demo'])
    shape = str(selected['shape'])
    episode = int(selected['episode'])
    spec = ARMS[tag]
    shape_idx = list(CONDITIONS[condition]['shapes']).index(shape)
    episode_seed = (31_000_000 + CONDITIONS[condition]['index'] * 1_000_000
                    + shape_idx * 10_000 + episode)
    cfg = CONDITIONS[condition]
    checkpoint = ROOT / 'models' / f'{tag}_seed{train_seed}.zip'
    if not checkpoint.is_file():
        raise FileNotFoundError(f'Missing selected checkpoint: {checkpoint}')

    print('[RECORD]', label, f'{shape=}, {train_seed=}, {episode=}', flush=True)
    env = None
    writer = None
    gif_frames = []
    gif_needed = DEMO_GIFS and label in {'A_train_success','A_train_failure','B_train_success','A_test2_success'}
    try:
        env = make_env_v3(spec, 0, eval_truth=True, shapes=cfg['shapes'],
                          size_mode=cfg['size_mode'],
                          mass_range=cfg.get('mass_range', (.05,.30)),
                          friction_range=cfg.get('friction_range', (.5,1.2)),
                          physics=cfg.get('physics'))
        agent_act = sb3_act(checkpoint, spec['algo'])
        obs, _ = env.reset(seed=episode_seed, options={'shape': shape})
        e = env.unwrapped
        writer = cv2.VideoWriter(str(target), cv2.VideoWriter_fourcc(*'mp4v'), DEMO_FPS, (640, 480))
        if not writer.isOpened():
            raise RuntimeError('MP4 writer unavailable. Check your OpenCV video codecs.')
        final = None
        for t in range(CFG.episode_steps):
            action = agent_act(env, obs, t)  # deterministic, as in final_eval
            obs, _, terminated, truncated, info = env.step(action)
            final = info
            if (t % DEMO_FRAME_EVERY == 0) or terminated or truncated:
                frame = _demo_frame(e, info, t+1, f'{label} | {shape} | SAC M2 seed {train_seed}')
                writer.write(frame)
                if gif_needed and t % (DEMO_FRAME_EVERY * 2) == 0:
                    small = cv2.resize(frame, (480, 360))
                    gif_frames.append(cv2.cvtColor(small, cv2.COLOR_BGR2RGB))
            if terminated or truncated:
                break
        assert final is not None
        actual_success = bool(final['gt_success_EVAL_ONLY'])
        outcome = {
            'label': label, 'policy': tag, 'train_seed': train_seed,
            'condition': condition, 'shape': shape, 'episode': episode,
            'episode_seed': episode_seed, 'expected_gt': int(want_success),
            'actual_gt': int(actual_success), 'actual_cv': int(final['cv_success']),
            'frames_simulated': t + 1, 'file': target.name,
        }
        metadata_path.write_text(json.dumps(outcome, indent=2), encoding='utf-8')
        if actual_success != want_success:
            print('[WARNING] Replay GT differs from final_eval CSV:', outcome)
        if gif_needed and gif_frames:
            imageio.mimsave(str(DEMO_DIR / f'{label}.gif'), gif_frames, duration=.2, loop=0)
        return outcome
    finally:
        if writer is not None:
            writer.release()
        if env is not None:
            env.close()
        gc.collect()

print('Demo output:', DEMO_DIR)
print('Prepared cases:', len(DEMO_CASES), '| train/test1/test2 plus Level B')


In [ ]:
# VisuoGrasp V3.3 — Cell 2: record demos from selected, saved SAC M2 checkpoints.
# Safe to rerun: finished MP4 files are skipped. Does not call model.learn().
results = []
for case in DEMO_CASES:
    try:
        result = record_v33_demo(case)
        if result is not None:
            results.append(result)
    except Exception as exc:
        print('[DEMO ERROR]', case[0], type(exc).__name__, str(exc))
        # Continue recording the other demo cases.

if results:
    demo_table = pd.DataFrame(results)
    demo_table.to_csv(DEMO_DIR / 'demo_manifest.csv', index=False)
    display(demo_table[['label','shape','condition','train_seed','expected_gt','actual_gt','actual_cv','file']])
print('Saved videos and GIFs in:', DEMO_DIR)
